# sub-16: recovering the bars sequences — constrained mglRetinotopy generator (v2)

**Previous notebook** (`sub16_stimulus_mgl.ipynb`):
- Controls: the true chains were recovered almost exactly. The score of the recovered chain was equal to that of the true stimulus (sub-16 LE02: 0.458 vs 0.454; sub-13 RE02: 0.574 vs 0.573). The few errors were 0°/45° confusions, and an offset of 1-2 TR of the start (absorbed by the HRF).
- The 3 suspect runs of sub-16 have the **structure of the A generator**: initial half-cycle blank, then sweeps with **exactly one inserted half-cycle blank**, and 7 distinct directions out of 8 (one repeat, likely one confusion). The order differs between runs. Score: 0.36-0.38 (vs 0.04-0.15 with A or B). For LE01, V1+V3 and V2 gave exactly the same chain.
- Null (rings): degenerate chains (the same direction repeated). Its score is high, so the score alone is not a proof; the structure is.

**This notebook** imposes the exact structure of the generator seen in A:
- an initial blank of 3-7 TR;
- **8 distinct directions** (a permutation) with **exactly one blank** of 5-6 TR inserted among them;
- free sweeps after the 8th, to fill the end of the run.

Beam search with this constraint, on V1+V3, on V2, and on all of V1-V3. A sequence is recovered if:
1. V1+V3 and V2 give the same order of directions and the same blank position;
2. the cross-validated score is at the level of the controls;
3. the best chain clearly beats the second best (margin).

Controls: sub-13 and sub-16 LE02 (A) must give the A order. sub-17 (B) does not follow this structure (no blank) and serves as a negative control of the constraint. Null: rings. Design matrices saved in `Desktop/fig_od/sub16_recovered_mgl_v2/`.

In [1]:
import os, glob, math
import numpy as np, pandas as pd, nibabel as nb
import scipy.io as sio

main = '/Users/uriel/disks/meso_shared/amblyo7T_prf'
pp = f'{main}/derivatives/pp_data'
out_dir = '/Users/uriel/Desktop/fig_od/sub16_recovered_mgl_v2'
os.makedirs(out_dir, exist_ok=True)
TR = 2.0
runs_to_do = [('sub-16', 'BarsLeftEye_run-01', 'suspect'), ('sub-16', 'BarsRightEye_run-01', 'suspect'),
              ('sub-16', 'BarsRightEye_run-02', 'suspect'), ('sub-16', 'BarsLeftEye_run-02', 'A'),
              ('sub-13', 'BarsLeftEye_run-01', 'A'), ('sub-13', 'BarsRightEye_run-02', 'A'),
              ('sub-17', 'BarsLeftEye_run-01', 'B'), ('sub-17', 'BarsRightEye_run-02', 'B'),
              ('sub-16', 'RingsLeftEye_run-01', 'null (rings)'), ('sub-16', 'RingsRightEye_run-01', 'null (rings)')]
beam_width = 250
n_vert_est = 600              # best vertices (pRF R2) of the estimation set used for the search
delays = [0, -1, -2]          # HRF delay tested (TR)
init_blank = range(3, 8)
ins_blank = [5, 6]

/var/folders/60/q71xm4hx45x3db5d6mzl8r4h0000gn/T/ipykernel_5539/2366921434.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import numpy as np, pandas as pd, nibabel as nb


## Known sequences and the 8 sweep templates

In [2]:
def build_vdm(mat_fn):
    """Same conversion as create_vdm_files.ipynb (50 x 50, transposed, padded)."""
    im = np.asarray(sio.loadmat(mat_fn, squeeze_me=True, struct_as_record=False)['s'].im, float)
    out = np.zeros((50, 50, im.shape[-1]))
    ii = np.floor(np.arange(50) * 102 / 50).astype(int)
    for t in range(im.shape[-1]):
        f = (im[..., t] * 255).astype(np.uint8).T
        sq = np.vstack([np.zeros((12, 102), np.uint8), f, np.zeros((13, 102), np.uint8)])
        out[..., t] = (sq > 5)[np.ix_(ii, ii)]
    return out

A = np.load(f'{main}/derivatives/vdm/task-BarsBarsRingsWedgesLeftEye_vdm.npy')[..., :120]
B = build_vdm(f'{main}/sourcedata/C/jp/cond-1-stim-3.mat')
true_stim = {'A': A, 'B': B}

rr, cc = np.meshgrid(np.arange(50) - 24.5, np.arange(50) - 24.5, indexing='ij')
X_pix, Y_pix = cc, -rr


def frame_label(f):
    m = f > 0
    if m.sum() < 3:
        return None, np.nan
    best = None
    for a in [0, 45, 90, 135]:
        th = np.deg2rad(a)
        p = X_pix[m] * np.cos(th) + Y_pix[m] * np.sin(th)
        if best is None or p.std() < best[0]:
            best = (p.std(), a, p.mean())
    return best[1], best[2]


def segment(stim):
    """stim (50,50,T) -> list of events (key, start, length); key = 'b' (blank) or '<ori><sign>'."""
    lab = [frame_label(stim[..., t]) for t in range(stim.shape[-1])]
    ev, t = [], 0
    while t < len(lab):
        o, p = lab[t]
        u = t + 1
        if o is None:
            while u < len(lab) and lab[u][0] is None: u += 1
            ev.append(('b', t, u - t))
        else:
            while u < len(lab) and lab[u][0] == o and u - t < 12: u += 1
            sign = '+' if lab[u - 1][1] > p else '-'
            ev.append((f'{o}{sign}', t, u - t))
        t = u
    return ev


templates = {}
for name, stim in [('A', A), ('B', B)]:
    ev = segment(stim)
    print(name, ' '.join(f'{k}({l})' for k, s, l in ev))
    for k, s, l in ev:
        if k != 'b' and l == 12 and k not in templates:
            templates[k] = stim[..., s:s + 12]
print('templates:', sorted(templates), '->', len(templates), 'directions')
true_events = {k: [(e[0], e[2]) for e in segment(v)] for k, v in true_stim.items()}

A b(6) 135+(12) 90+(12) b(5) 45-(12) 0+(12) 135-(12) 0-(12) 45+(12) 90-(12) 135-(12) 135-(1)
B 0+(12) 135-(12) 90-(12) 45-(12) 0-(12) 135+(12) 90+(12) 45+(12) 0+(12) 135-(12)
templates: ['0+', '0-', '135+', '135-', '45+', '45-', '90+', '90-'] -> 8 directions


## Data, pRFs and prediction

In [3]:
cache = {}
def load_subject(s):
    if s in cache: return cache[s]
    roi = np.concatenate([nb.load(glob.glob(f'{pp}/{s}/fsnative/rois/{s}_hemi-{h}_fmriprep_dct_*_rois-group-mmp.func.gii')[0]).darrays[0].data
                          for h in ['L', 'R']])
    prf = {eye: np.hstack([np.vstack([a.data for a in nb.load(
        f'{pp}/{s}/fsnative/prfmodel/prf_derivatives/{s}_task-RingsWedges{eye}_hemi-{h}_fmriprep_dct_psc_concat_prf-css_deriv.func.gii').darrays])
        for h in ['L', 'R']]) for eye in ['LeftEye', 'RightEye']}
    cache[s] = (roi, prf)
    return cache[s]


def load_run(s, r):
    hs = []
    for h in ['L', 'R']:
        d = np.vstack([a.data for a in nb.load(
            f'{pp}/{s}/fsnative/func/fmriprep_dct_psc/{s}_ses-01_task-{r}_hemi-{h}_space-fsnative_fmriprep_dct_psc_bold.func.gii').darrays])
        hs.append(d if d.shape[0] == 120 else d.T)
    return np.hstack(hs)


pix = 2 * math.degrees(math.atan(50.6 / 2 / 242)) / 50
Xd, Yd = X_pix * pix, Y_pix * pix
t_ = np.arange(0, 32, TR)
g = lambda a: t_ ** (a - 1) * np.exp(-t_) / math.gamma(a)
H = g(6) - g(16) / 6; H /= H.sum()
N_fft = 256
Hf = np.fft.rfft(H, n=N_fft)


def conv_hrf(d):
    return np.fft.irfft(np.fft.rfft(d, n=N_fft, axis=0) * Hf[:, None], n=N_fft, axis=0)[:d.shape[0]]


def shift(p, sh):
    q = np.roll(p, sh, 0)
    if sh > 0: q[:sh] = 0
    elif sh < 0: q[sh:] = 0
    return q


def zc(a, axis=0):
    a = a - a.mean(axis, keepdims=True)
    return a / (a.std(axis, keepdims=True) + 1e-9)


def drive(frames, x0, y0, s, n):
    """frames (k,50,50) -> (k, vertices)."""
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * s ** 2))
    return np.clip(frames.reshape(len(frames), -1) @ G, 0, None) ** n


def events_to_stim(events):
    st = np.zeros((50, 50, 120))
    t = 0
    for k, l in events:
        if k != 'b' and t < 120:
            n = min(12, 120 - t)
            st[..., t:t + n] = templates[k][..., :n]
        t += l
    return st


def test_score(stim, y, D_test):
    """Median correlation between data and prediction of the test vertices, best delay among -2..+2 TR."""
    x0, y0, s, n = D_test
    p = conv_hrf(drive(np.moveaxis(stim, -1, 0), x0, y0, s, n))
    best = -np.inf
    for sh in range(-2, 3):
        q = np.roll(p, sh, 0)
        if sh > 0: q[:sh] = p[0]
        elif sh < 0: q[sh:] = p[-1]
        best = max(best, float(np.median((zc(q) * zc(y)).mean(0))))
    return best

## Constrained beam search

In [4]:
def window_score(pred, y, w):
    p, q = pred[:w], y[:w]
    p = p - p.mean(0); q = q - q.mean(0)
    den = np.sqrt((p ** 2).sum(0) * (q ** 2).sum(0))
    r = np.where(den > 0, (p * q).sum(0) / np.where(den > 0, den, 1), 0)
    return float(r.mean())


def order_key(ev):
    """Order of the directions of the permutation + index of the inserted blank (timing-free)."""
    seq, blank_at = [], None
    for k, l in ev[1:]:
        if k == 'b': blank_at = len(seq)
        else: seq.append(k)
    return tuple(seq[:8]), blank_at


def ev_str(ev):
    return ' '.join(f'b{l}' if k == 'b' else k for k, l in ev)


def search(y, D_est, delay, n_keep=20):
    """Constrained search. Returns the n_keep best distinct chains (events, score on the full run)."""
    x0, y0, s, n = D_est
    kern = {}
    for k, fr in templates.items():
        z = np.zeros((120, len(x0)), np.float32)
        z[:12] = drive(np.moveaxis(fr, -1, 0), x0, y0, s, n)
        kern[k] = shift(conv_hrf(z), delay).astype(np.float32)
    y = y.astype(np.float32)
    # state: pred, t, events, used directions, inserted blank done
    beam = [(np.zeros_like(y), l, [('b', l)], frozenset(), False) for l in init_blank]
    done = []
    while beam:
        cand = []
        for pred, t, ev, used, bl in beam:
            n_sw = len(used)
            if n_sw < 8:
                opts = [(k, 12) for k in kern if k not in used]
                if not bl and n_sw >= 1 and ev[-1][0] != 'b':
                    opts += [('b', l) for l in ins_blank]
            else:
                opts = [(k, 12) for k in kern]          # after the permutation: free
            for k, l in opts:
                p2 = pred
                if k != 'b':
                    p2 = pred.copy(); p2[t:] += kern[k][:120 - t]
                used2 = (used | {k}) if (k != 'b' and n_sw < 8) else used
                bl2 = bl or (k == 'b')
                t2, ev2 = t + l, ev + [(k, l)]
                if t2 >= 120:
                    if len(used2) >= 7:                 # the run may end before the 8th sweep is complete
                        done.append((ev2, window_score(p2, y, 120)))
                else:
                    cand.append((p2, t2, ev2, used2, bl2, window_score(p2, y, min(120, t2 + 3))))
        cand.sort(key=lambda z: -z[5])
        beam = [z[:5] for z in cand[:beam_width]]
    done.sort(key=lambda z: -z[1])
    out, seen = [], set()
    for ev, sc in done:
        key = order_key(ev)
        if key not in seen:
            seen.add(key); out.append((ev, sc))
        if len(out) >= n_keep: break
    return out


In [5]:
true_key = {k: order_key(v) for k, v in true_events.items()}
rows, recovered = [], {}
for s, r, kind in runs_to_do:
    roi, prf = load_subject(s)
    eye = 'LeftEye' if 'LeftEye' in r else 'RightEye'
    D = prf[eye]
    y_all = load_run(s, r)
    ok = (D[0] > 0.2) & (D[1] < 6) & np.isfinite(y_all).all(0) & (y_all.std(0) > 0)
    sets = {'V1V3': np.where(ok & ((roi == 1) | (roi == 3)))[0], 'V2': np.where(ok & (roi == 2))[0]}
    sets['all'] = np.concatenate([sets['V1V3'], sets['V2']])
    par = lambda idx: (D[7][idx], D[8][idx], np.clip(D[4][idx], 0.2, None), np.clip(D[11][idx], 0.05, 1))
    res = {'subject': s, 'run': r, 'true': kind}
    best = {}
    for est in ['V1V3', 'V2', 'all']:
        idx = sets[est][np.argsort(-D[0][sets[est]])[:n_vert_est]]
        yz = zc(y_all[:, idx])
        tops = [(search(yz, par(idx), dl), dl) for dl in delays]
        lst, dl = max(tops, key=lambda z: z[0][0][1])
        best[est] = (lst, dl)
        res[f'chain {est}'] = ev_str(lst[0][0])
    res['same order V1V3 / V2'] = order_key(best['V1V3'][0][0][0]) == order_key(best['V2'][0][0][0])
    lst_all = best['all'][0]
    res['margin best - 2nd (all)'] = lst_all[0][1] - lst_all[1][1]
    k_all = order_key(lst_all[0][0])
    res['order = A'] = k_all == true_key['A']
    res['n directions identical to A order'] = sum(a == b for a, b in zip(k_all[0], true_key['A'][0]))
    res['cv r recovered'] = np.mean([test_score(events_to_stim(best[e][0][0][0]), y_all[:, sets[t]], par(sets[t]))
                                     for e, t in [('V1V3', 'V2'), ('V2', 'V1V3')]])
    for k, v in true_stim.items():
        res[f'cv r true {k}'] = np.mean([test_score(v, y_all[:, sets[t]], par(sets[t])) for t in ['V2', 'V1V3']])
    res['delay (all)'] = best['all'][1]
    stim_rec = events_to_stim(lst_all[0][0])
    recovered[(s, r)] = (best, stim_rec)
    np.save(f'{out_dir}/{s}_{r}_recovered_vdm.npy', stim_rec)
    rows.append(res)
    print(s, r, 'done', flush=True)
tab = pd.DataFrame(rows).set_index(['subject', 'run'])
pd.set_option('display.max_colwidth', 200)
print('true A:', ev_str(true_events['A']))
tab[['true', 'chain V1V3', 'chain V2', 'chain all']]


sub-16 BarsLeftEye_run-01 done
sub-16 BarsRightEye_run-01 done
sub-16 BarsRightEye_run-02 done
sub-16 BarsLeftEye_run-02 done
sub-13 BarsLeftEye_run-01 done
sub-13 BarsRightEye_run-02 done
sub-17 BarsLeftEye_run-01 done
sub-17 BarsRightEye_run-02 done
sub-16 RingsLeftEye_run-01 done
sub-16 RingsRightEye_run-01 done
true A: b6 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135-


true  \
subject run                                  
sub-16  BarsLeftEye_run-01         suspect   
        BarsRightEye_run-01        suspect   
        BarsRightEye_run-02        suspect   
        BarsLeftEye_run-02               A   
sub-13  BarsLeftEye_run-01               A   
        BarsRightEye_run-02              A   
sub-17  BarsLeftEye_run-01               B   
        BarsRightEye_run-02              B   
sub-16  RingsLeftEye_run-01   null (rings)   
        RingsRightEye_run-01  null (rings)   

                                                                   chain V1V3  \
subject run                                                                     
sub-16  BarsLeftEye_run-01        b4 90+ 45+ 0- b6 0+ 45- 135+ 90- 135- 0- 0+   
        BarsRightEye_run-01         b7 90- 0+ 135- 45- 135+ 90+ b5 0- 45+ 90-   
        BarsRightEye_run-02       b4 0+ 0- 45+ 135+ 45- b6 135- 90+ 90- 0+ 0+   
        BarsLeftEye_run-02      b4 135+ 90+ b6 45- 0+ 135- 0- 45+ 90- 135- 0-   
sub-13  BarsLeftEye_run-01    b5 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135+   
        BarsRightEye_run-02     b5 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 0-   
sub-17  BarsLeftEye_run-01    b6 90+ b5 90- 45- 0- 135+ 45+ 0+ 135- 135- 135+   
        BarsRightEye_run-02   b6 90+ b5 90- 45- 0- 135+ 45+ 0+ 135- 135- 135+   
sub-16  RingsLeftEye_run-01     b6 0- b5 135+ 45+ 135- 0+ 90+ 90- 45- 135+ 0-   
        RingsRightEye_run-01    b3 90+ b6 45+ 0- 0+ 135- 90- 135+ 45- 135- 0-   

                                                                     chain V2  \
subject run                                                                     
sub-16  BarsLeftEye_run-01        b4 90+ 0+ b6 45+ 90- 0- 135+ 45- 135- 0- 0+   
        BarsRightEye_run-01         b7 135- 0+ 90- 45- 135+ 90+ b5 0- 45+ 90-   
        BarsRightEye_run-02      b5 0+ 0- 45+ 135+ 45- b5 135- 90+ 90- 0+ 90+   
        BarsLeftEye_run-02    b4 135+ 90+ b6 45- 0+ 135- 0- 45+ 90- 135- 135-   
sub-13  BarsLeftEye_run-01     b5 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 45-   
        BarsRightEye_run-02     b4 135+ 90+ b6 45- 0+ 135- 0- 45+ 90- 135- 0-   
sub-17  BarsLeftEye_run-01    b6 135+ b5 90- 45- 0- 90+ 45+ 0+ 135- 135- 135+   
        BarsRightEye_run-02   b6 90+ b5 90- 45- 0- 135+ 45+ 0+ 135- 135- 135+   
sub-16  RingsLeftEye_run-01       b5 45- b5 135+ 45+ 90- 0+ 135- 0- 90+ 0+ 0-   
        RingsRightEye_run-01      b3 135- b6 90- 135+ 45+ 45- 0- 90+ 0+ 0- 0-   

                                                                    chain all  
subject run                                                                    
sub-16  BarsLeftEye_run-01        b4 90+ 0+ b6 45+ 135- 0- 135+ 45- 90- 0- 0+  
        BarsRightEye_run-01         b7 90- 0+ 135- 45- 135+ 90+ b5 0- 45+ 90-  
        BarsRightEye_run-02       b4 0+ 0- 45+ 135+ 45- b6 135- 90+ 90- 0+ 0+  
        BarsLeftEye_run-02      b4 135+ 90+ b6 45- 0+ 135- 0- 45+ 90- 135- 0-  
sub-13  BarsLeftEye_run-01      b5 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 0-  
        BarsRightEye_run-02     b5 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 0-  
sub-17  BarsLeftEye_run-01    b6 90+ b5 90- 45- 0- 135+ 45+ 0+ 135- 135- 135+  
        BarsRightEye_run-02   b6 90+ b5 90- 45- 0- 135+ 45+ 0+ 135- 135- 135+  
sub-16  RingsLeftEye_run-01       b3 90+ b6 0- 45+ 45- 135- 90- 135+ 0+ 0- 0-  
        RingsRightEye_run-01    b3 90+ b6 90- 135+ 0+ 135- 0- 45+ 45- 135- 0-

## Decision

In [6]:
cols = ['true', 'same order V1V3 / V2', 'order = A', 'n directions identical to A order', 'margin best - 2nd (all)',
        'cv r recovered', 'cv r true A', 'cv r true B', 'delay (all)']
print(tab[cols].round(4).to_string())
print()
print("How to read:")
print("- Controls A (sub-13, sub-16 LE02): order = A True, same order V1V3 / V2, cv r recovered close to cv r true A.")
print("- sub-17 (B, no blank) and rings do not follow this structure: lower cv r recovered than without constraint, unstable orders.")
print("- Suspect runs: recovered if same order V1V3 / V2 and cv r at the level of the A control of sub-16 (LE02).")
print("  The margin tells how much the best chain beats the 2nd: compare it with that of the A controls.")
print("Design matrices (from the 'all' search) saved in", out_dir)
print()
for (s, r), (best, _) in recovered.items():
    if s == 'sub-16' and 'Bars' in r:
        print(s, r, '| 5 best chains (all):')
        for ev, sc in best['all'][0][:5]:
            print('   {:.4f}  {}'.format(sc, ev_str(ev)))


                                      true  same order V1V3 / V2  order = A  n directions identical to A order  margin best - 2nd (all)  cv r recovered  cv r true A  cv r true B  delay (all)
subject run                                                                                                                                                                                   
sub-16  BarsLeftEye_run-01         suspect                 False      False                                  1                   0.0006          0.3310       0.1463       0.0546            0
        BarsRightEye_run-01        suspect                 False      False                                  0                   0.0116          0.3532       0.0400       0.0027           -2
        BarsRightEye_run-02        suspect                  True      False                                  1                   0.0020          0.3467       0.0422       0.0213            0
        BarsLeftEye_run-02               A   